# House Price Prediction
Configure the housing CSV and model parameters in `conf/config.yml`, then run this notebook from `notebooks/housing_price`.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from ta_lib.custom import SignedLog1pTransformer

with Path('conf/config.yml').open() as config_file:
    config = yaml.safe_load(config_file)
housing = pd.read_csv(Path(config['dataset_path']))
target = config['target_column']
X = housing.drop(columns=[target])
y = housing[target]

In [ ]:
numeric = X.select_dtypes(include=['number']).columns.tolist()
categorical = X.select_dtypes(exclude=['number']).columns.tolist()
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('signed_log1p', SignedLog1pTransformer()),
    ]), numeric),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical),
])
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        random_state=config['random_state'], n_jobs=-1, **config['model']
    )),
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config['test_size'], random_state=config['random_state']
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
pd.Series({
    'mae': mean_absolute_error(y_test, predictions),
    'rmse': mean_squared_error(y_test, predictions) ** 0.5,
    'r2': r2_score(y_test, predictions),
}, name='value')